# Legibility models trained on handwritten words, tested on signatures

**Design (fixed before any holdout scoring)**

| Stage | Data | Used for |
|---|---|---|
| Train | IMGUR5K train words (length-balanced) | fitting every model |
| In-domain check | IMGUR5K val words | does the model learn word legibility at all |
| Model selection | HF dev signatures (1,035) | choosing the best model **for signatures** |
| Final holdout | HF test signatures (254, 23 signers) | scored once, after the choice is locked |

**Pre-registered decisions** (from the word data only): similarity threshold 0.8 for words and signatures; words with 4+ letters; light-on-dark crops dropped (dark-pixel share > 0.35); training words rebalanced so every length group has the same legible rate, so word length cannot predict the label.

**Added 2026-11-10, before any HF-test scoring:** CNN variants trained on the TrOCR similarity itself instead of the 0/1 label (`raw`: target = similarity; `soft T0.05` / `soft T0.10`: target = sigmoid((similarity − 0.8) / T)). They enter the same selection rule (max HF-dev PR-AUC) and are scored against the same hard 0.8 labels.

**Baselines that every model must beat:** majority class, contrast only, and length only (letters in the intended word/name).

Needs in this folder: `model_comparison.py`, `word_legibility_pipeline.py`, `words/words_256.npy`, `words/words_labels.csv`, `signature_images_256.npy`, `signatures_labels.csv`, `splits.csv`. The CNN cells need PyTorch.

## 1. Setup

In [2]:
import json, time, warnings
from pathlib import Path
import numpy as np, pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import average_precision_score, roc_auc_score, f1_score
import sys
from pathlib import Path
PY_DIR = Path("PY files").resolve()
assert (PY_DIR / "model_comparison.py").exists(), f"model_comparison.py not found in {PY_DIR}"
sys.path.insert(0, str(PY_DIR))
import model_comparison as mc            # same features and models as the signature study
from word_legibility_pipeline import clean_gray   # same cleaning used before OCR labeling
warnings.filterwarnings("ignore")

CFG = dict(
    img_size=128,          # CNN / HOG input size (same as the signature study)
    epochs=10,             # fixed, no early stopping on any evaluation set
    batch_size=64,
    no_amp=False,
    cnn_seeds=(42, 43, 44),
    svm_max_train=10000,   # RBF SVM is O(n^2); trained on a random subsample
    rf_trees=300,
    finetune_epochs=8,
    dark_frac_max=0.35,
    seed=42,
)
rng = np.random.default_rng(CFG["seed"])

def letters_count(s):
    return pd.Series(s).astype(str).str.replace(r"[^A-Za-z]", "", regex=True).str.len().to_numpy()

def metrics(y, s, thr):
    return {"n": len(y), "prevalence": round(float(np.mean(y)), 3),
            "pr_auc": average_precision_score(y, s), "roc_auc": roc_auc_score(y, s),
            "f1": f1_score(y, (np.asarray(s) > thr).astype(int), zero_division=0)}

## 2. Load words and signatures

In [3]:
# ---- words ----
W_ARR = np.load("words/words_256.npy", mmap_mode="r")
wl = pd.read_csv("words/words_labels.csv")
wl = wl[wl["similarity"].notna()].copy()
wl["n_letters"] = letters_count(wl["target"])
wl["dark_frac"] = [(np.asarray(W_ARR[int(i)]) < 128).mean() for i in wl["idx"]]
dropped = int((wl["dark_frac"] > CFG["dark_frac_max"]).sum())
wl = wl[(wl["dark_frac"] <= CFG["dark_frac_max"]) & (wl["n_letters"] >= 4)]
print(f"words labeled: {len(wl)}  (light-on-dark dropped: {dropped})")
print(wl.groupby("split")["label"].agg(["size", "mean"]).rename(columns={"mean": "legible"}).round(3))

# ---- signatures (labels from the same OCR procedure, images cleaned the same way) ----
splits = pd.read_csv("splits.csv")
sl = pd.read_csv("signatures_labels.csv").sort_values("idx").reset_index(drop=True)
assert len(sl) == len(splits) and (sl["idx"].to_numpy() == np.arange(len(splits))).all()
sl["split"], sl["group"], sl["family"] = splits["split"], splits["group"], splits["source_family"]
sl["n_letters"] = letters_count(splits["name"])
S_RAW = np.load("signature_images_256.npy", mmap_mode="r")
S_CLEAN = np.stack([clean_gray(np.asarray(g)) for g in S_RAW])
dev, test = sl[sl.split == "dev"], sl[sl.split == "test"]
print(f"\nsignatures: dev {len(dev)} (legible {dev.label.mean():.3f}), "
      f"test {len(test)} (legible {test.label.mean():.3f})")

words labeled: 49848  (light-on-dark dropped: 148)
        size  legible
split                
test    4995    0.513
train  39878    0.517
val     4975    0.502

signatures: dev 1035 (legible 0.293), test 254 (legible 0.311)


## 3. Length-balance the training words
Within each length group (4, 5, …, 10+ letters) the majority class is randomly downsampled so every group ends up with the same legible rate. After this, word length carries no information about the label in the training set.

In [4]:
tr = wl[wl.split == "train"].copy()
tr["len_bin"] = tr["n_letters"].clip(upper=10)
target = tr["label"].mean()
keep = []
for b, g in tr.groupby("len_bin"):
    pos, neg = g[g.label == 1], g[g.label == 0]
    n_pos = min(len(pos), int(round(len(neg) * target / (1 - target))))
    n_neg = min(len(neg), int(round(n_pos * (1 - target) / target)))
    keep.append(pos.sample(n_pos, random_state=CFG["seed"]))
    keep.append(neg.sample(n_neg, random_state=CFG["seed"]))
train = pd.concat(keep).sample(frac=1, random_state=CFG["seed"]).reset_index(drop=True)
val = wl[wl.split == "val"].reset_index(drop=True)
print(f"train words: {len(tr)} -> {len(train)} after balancing; target legible rate {target:.3f}")
print(train.groupby("len_bin")["label"].agg(["size", "mean"]).rename(columns={"mean": "legible"}).round(3))

train words: 39878 -> 35666 after balancing; target legible rate 0.517
          size  legible
len_bin                
4        11565    0.517
5         8051    0.517
6         5646    0.517
7         4402    0.517
8         2517    0.517
9         1585    0.517
10        1900    0.517


## 3b. Soft targets from the TrOCR similarity

The hard label throws away how close a read was: 0.79 and 0.20 both become 0, 0.81 and 1.00 both become 1. These targets keep that information:

| mode | target | e.g. 0.75 → | 0.85 → |
|---|---|---|---|
| `hard` | 1 if similarity ≥ 0.8 | 0 | 1 |
| `raw` | similarity | 0.75 | 0.85 |
| `soft_T0.05` | sigmoid((sim − 0.8)/0.05) | 0.27 | 0.73 |
| `soft_T0.10` | sigmoid((sim − 0.8)/0.10) | 0.38 | 0.62 |

Evaluation still uses the hard 0.8 labels, so every number stays comparable.

In [5]:
SOFT_MODES = ["raw", "soft_T0.05", "soft_T0.10"]
SIG_SIM_COL = "similarity"        # similarity column in signatures_labels.csv

def make_target(sim, mode, thr=0.8):
    sim = np.asarray(sim, dtype=np.float32)
    if sim.max() > 1.0:           # stored as a percentage
        sim = sim / 100.0
    if mode == "raw":
        return sim.clip(0, 1)
    if mode.startswith("soft_T"):
        T = float(mode.split("_T")[1])
        return (1.0 / (1.0 + np.exp(-(sim - thr) / T))).astype(np.float32)
    raise ValueError(mode)

def mode_label(mode):
    return {"raw": "raw similarity", "soft_T0.05": "soft T0.05", "soft_T0.10": "soft T0.10"}[mode]
WORD_NAME = {"hard": "CNN (words)", **{m: f"CNN (words, {mode_label(m)})" for m in SOFT_MODES}}
FT_NAME   = {"hard": "CNN (words + signature fine-tune)",
             **{m: f"CNN (words + signature fine-tune, {mode_label(m)})" for m in SOFT_MODES}}

assert SIG_SIM_COL in sl.columns, f"set SIG_SIM_COL to one of {list(sl.columns)}"
T_TRAIN = {m: make_target(train["similarity"], m) for m in SOFT_MODES}
T_DEV   = {m: make_target(dev[SIG_SIM_COL], m) for m in SOFT_MODES}
T_TRAIN["hard"], T_DEV["hard"] = train["label"].to_numpy(), dev["label"].to_numpy()

# the stored labels should be exactly similarity >= 0.8
for nm, frame, col in [("words", train, "similarity"), ("signatures (dev)", dev, SIG_SIM_COL)]:
    s = frame[col].to_numpy(float); s = s / 100 if s.max() > 1 else s
    agree = ((s >= 0.8).astype(int) == frame["label"].to_numpy()).mean()
    print(f"{nm}: label == (similarity >= 0.8) for {agree:.4f} of rows")

print(pd.DataFrame({m: make_target(np.array([.2, .5, .7, .75, .79, .81, .85, .9, 1.]), m) for m in SOFT_MODES},
                   index=[.2, .5, .7, .75, .79, .81, .85, .9, 1.]).round(2))

words: label == (similarity >= 0.8) for 1.0000 of rows
signatures (dev): label == (similarity >= 0.8) for 1.0000 of rows
       raw  soft_T0.05  soft_T0.10
0.20  0.20        0.00        0.00
0.50  0.50        0.00        0.05
0.70  0.70        0.12        0.27
0.75  0.75        0.27        0.38
0.79  0.79        0.45        0.48
0.81  0.81        0.55        0.52
0.85  0.85        0.73        0.62
0.90  0.90        0.88        0.73
1.00  1.00        0.98        0.88


### Does the soft target bring word length back in?

Length balancing made the **hard** label independent of length. A soft target could still carry length information: with Levenshtein similarity, one wrong letter costs a 4-letter word 0.25 but a 10-letter word only 0.10. If the mean target differs a lot across length bins *within* each label, the soft models can pick up some length signal. The length-only baseline on HF dev is the safeguard, but it's worth knowing now.

In [6]:
from scipy.stats import spearmanr
chk = train[["len_bin", "label", "n_letters"]].copy()
for m in SOFT_MODES:
    chk[m] = T_TRAIN[m]
print("mean target by length bin and hard label:")
display(chk.groupby(["len_bin", "label"])[SOFT_MODES].mean().unstack("label").round(3))
rows = []
for m in SOFT_MODES:
    r = {"mode": m, "spearman(length, target) overall": spearmanr(chk.n_letters, chk[m]).statistic}
    for lab, g in chk.groupby("label"):
        r[f"within label={lab}"] = spearmanr(g.n_letters, g[m]).statistic
    rows.append(r)
display(pd.DataFrame(rows).round(3))
print("|rho| < 0.1 within each label: negligible length leakage")

mean target by length bin and hard label:


raw        soft_T0.05        soft_T0.10       
label        0      1          0      1          0      1
len_bin                                                  
4        0.361  0.982      0.037  0.938      0.076  0.846
5        0.329  0.968      0.012  0.912      0.050  0.822
6        0.344  0.971      0.025  0.931      0.064  0.830
7        0.357  0.977      0.035  0.949      0.074  0.843
8        0.339  0.975      0.036  0.945      0.072  0.840
9        0.329  0.968      0.033  0.930      0.065  0.827
10       0.301  0.968      0.023  0.931      0.052  0.828

,mode,"spearman(length, target) overall",within label=0,within label=1
0,raw,-0.028,-0.043,-0.099
1,soft_T0.05,-0.028,-0.043,-0.099
2,soft_T0.10,-0.028,-0.043,-0.099


|rho| < 0.1 within each label: negligible length leakage


## 4. Features (same HOG + shape features and image variants as the signature study)

In [7]:
def norm_variant(arr_like, idx):
    out = np.zeros((len(idx), CFG["img_size"], CFG["img_size"]), np.uint8)
    for k in range(0, len(idx), 2000):          # chunks keep memory low
        chunk = np.stack([np.asarray(arr_like[int(i)]) for i in idx[k:k + 2000]])
        out[k:k + 2000] = mc.make_variant(chunk, "norm", CFG["img_size"])
    return out

t0 = time.time()
X = {"train": norm_variant(W_ARR, train["idx"].to_numpy()),
     "val":   norm_variant(W_ARR, val["idx"].to_numpy()),
     "dev":   norm_variant(S_CLEAN, dev.index.to_numpy()),
     "test":  norm_variant(S_CLEAN, test.index.to_numpy())}
Y = {"train": train.label.to_numpy(), "val": val.label.to_numpy(),
     "dev": dev.label.to_numpy(), "test": test.label.to_numpy()}
LEN = {"train": train.n_letters.to_numpy(), "val": val.n_letters.to_numpy(),
       "dev": dev.n_letters.to_numpy(), "test": test.n_letters.to_numpy()}
F = {k: mc.classical_features(v) for k, v in X.items()}
def contrast_chunked(arr_like, idx, chunk=2000):
    out = []
    for k in range(0, len(idx), chunk):
        block = np.stack([np.asarray(arr_like[int(i)]) for i in idx[k:k + chunk]])
        out.append(mc.contrast_feature(block))
    return np.concatenate(out)

C = {"train": contrast_chunked(W_ARR, train["idx"].to_numpy()),
     "val":   contrast_chunked(W_ARR, val["idx"].to_numpy()),
     "dev":   contrast_chunked(S_CLEAN, dev.index.to_numpy()),
     "test":  contrast_chunked(S_CLEAN, test.index.to_numpy())}
print({k: v.shape for k, v in F.items()}, f"{time.time() - t0:.0f}s")

{'train': (35666, 1772), 'val': (4975, 1772), 'dev': (1035, 1772), 'test': (254, 1772)} 126s


## 5. Train on words; score words-val and HF dev
HF test is **not** scored here.

In [8]:
SCORES = {}            # model -> {"val": s, "dev": s, "test": s}; test is filled only in section 8
THR = {}
FITTED = {}

def fit_score(name, model, Xtr, ytr, sets):
    t = time.time()
    model.fit(Xtr, ytr)
    FITTED[name] = (model, sets)
    SCORES[name] = {}
    for k in ("val", "dev"):
        SCORES[name][k], THR[name] = mc.score_and_threshold(model, sets[k])
    print(f"  {name:28s} {time.time() - t:6.0f}s")

models = mc.classical_models()
fit_score("Majority baseline", models["Majority baseline"](), F["train"], Y["train"], F)
fit_score("Logistic (contrast only)", models["Logistic (contrast only)"](), C["train"], Y["train"], C)
L = {k: v.reshape(-1, 1).astype(float) for k, v in LEN.items()}
fit_score("Logistic (length only)", make_pipeline(StandardScaler(), LogisticRegression()), L["train"], Y["train"], L)
fit_score("Logistic regression", models["Logistic regression"](), F["train"], Y["train"], F)
rf = models["Random forest"](); rf.set_params(n_estimators=CFG["rf_trees"])
fit_score("Random forest", rf, F["train"], Y["train"], F)
fit_score("Gradient boosting", models["Gradient boosting"](), F["train"], Y["train"], F)
sub = rng.permutation(len(Y["train"]))[:CFG["svm_max_train"]]
fit_score("SVM (RBF)", models["SVM (RBF)"](), F["train"][sub], Y["train"][sub], F)

  Majority baseline                 0s
  Logistic (contrast only)          0s
  Logistic (length only)            0s
  Logistic regression               4s
  Random forest                    23s
  Gradient boosting               259s
  SVM (RBF)                       116s


In [9]:
# ---- CNN trained on words (one model per seed; also kept for fine-tuning) ----
try:
    import torch
    HAVE_TORCH = True
except ImportError:
    HAVE_TORCH = False
    print("PyTorch not installed: CNN cells are skipped.")

if HAVE_TORCH:
    import torch.nn as nn
    DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print("device:", DEVICE)

    def build_cnn():
        def block(i, o):
            return nn.Sequential(nn.Conv2d(i, o, 3, padding=1, bias=False), nn.BatchNorm2d(o),
                                 nn.ReLU(inplace=True), nn.MaxPool2d(2))
        return nn.Sequential(block(1, 16), block(16, 32), block(32, 64), block(64, 128),
                             nn.AdaptiveAvgPool2d(1), nn.Flatten(), nn.Dropout(0.3),
                             nn.Linear(128, 1)).to(DEVICE)

    def train_cnn(model, Xtr, ytr, epochs, seed, max_lr=3e-3):
        torch.manual_seed(seed); np.random.seed(seed)
        pos = max(1, int(ytr.sum()))
        loss_fn = nn.BCEWithLogitsLoss(pos_weight=torch.tensor([(len(ytr) - pos) / pos], device=DEVICE))
        opt = torch.optim.AdamW(model.parameters(), lr=max_lr / 3, weight_decay=1e-4)
        steps = epochs * int(np.ceil(len(ytr) / CFG["batch_size"]))
        sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=max_lr, total_steps=steps)
        amp = DEVICE.type == "cuda" and not CFG["no_amp"]
        scaler = torch.amp.GradScaler("cuda", enabled=amp)
        Xt, yt = torch.from_numpy(Xtr), torch.from_numpy(ytr.astype(np.float32))
        for ep in range(epochs):
            model.train(); perm = torch.randperm(len(yt)); tot = 0.0
            for s in range(0, len(perm), CFG["batch_size"]):
                idx = perm[s:s + CFG["batch_size"]]
                if len(idx) < 2: continue
                xb = (Xt[idx].to(DEVICE).float() / 255).unsqueeze(1)
                yb = yt[idx].to(DEVICE)
                opt.zero_grad(set_to_none=True)
                with torch.autocast(device_type=DEVICE.type, enabled=amp):
                    loss = loss_fn(model(xb).squeeze(1), yb)
                scaler.scale(loss).backward(); scaler.step(opt); scaler.update(); sched.step()
                tot += loss.item() * len(idx)
            print(f"    epoch {ep + 1}/{epochs} loss {tot / len(yt):.4f}")
        return model

    @torch.no_grad()
    def predict_cnn(model, Xs):
        model.eval(); out = []
        amp = DEVICE.type == "cuda" and not CFG["no_amp"]
        for s in range(0, len(Xs), 256):
            xb = (torch.from_numpy(Xs[s:s + 256]).to(DEVICE).float() / 255).unsqueeze(1)
            with torch.autocast(device_type=DEVICE.type, enabled=amp):
                out.append(torch.sigmoid(model(xb).squeeze(1)).float().cpu().numpy())
        return np.concatenate(out)

    CNN_WORD = {}
    for sd in CFG["cnn_seeds"]:
        t = time.time(); print(f"CNN seed {sd}")
        m = train_cnn(build_cnn(), X["train"], Y["train"], CFG["epochs"], sd)
        CNN_WORD[sd] = {k: v.detach().cpu().clone() for k, v in m.state_dict().items()}
        name = f"CNN (words)__s{sd}"
        SCORES[name] = {k: predict_cnn(m, X[k]) for k in ("val", "dev")}; THR[name] = 0.5
        print(f"  done in {time.time() - t:.0f}s")
        del m
        if DEVICE.type == "cuda": torch.cuda.empty_cache()

device: cuda
CNN seed 42
    epoch 1/10 loss 0.5804
    epoch 2/10 loss 0.5440
    epoch 3/10 loss 0.5238
    epoch 4/10 loss 0.5118
    epoch 5/10 loss 0.4988
    epoch 6/10 loss 0.4910
    epoch 7/10 loss 0.4789
    epoch 8/10 loss 0.4690
    epoch 9/10 loss 0.4564
    epoch 10/10 loss 0.4465
  done in 88s
CNN seed 43
    epoch 1/10 loss 0.5760
    epoch 2/10 loss 0.5435
    epoch 3/10 loss 0.5298
    epoch 4/10 loss 0.5130
    epoch 5/10 loss 0.5008
    epoch 6/10 loss 0.4906
    epoch 7/10 loss 0.4791
    epoch 8/10 loss 0.4686
    epoch 9/10 loss 0.4565
    epoch 10/10 loss 0.4484
  done in 82s
CNN seed 44
    epoch 1/10 loss 0.5792
    epoch 2/10 loss 0.5428
    epoch 3/10 loss 0.5249
    epoch 4/10 loss 0.5104
    epoch 5/10 loss 0.5007
    epoch 6/10 loss 0.4916
    epoch 7/10 loss 0.4800
    epoch 8/10 loss 0.4683
    epoch 9/10 loss 0.4556
    epoch 10/10 loss 0.4492
  done in 83s


### Soft-target CNNs on words
Same architecture, epochs, seeds and optimizer as `CNN (words)`; only the target changes. About 90 s per model on your GPU → ~14 min for 3 modes × 3 seeds.

In [10]:
CNN_STATES = {"hard": CNN_WORD} if HAVE_TORCH else {}
if HAVE_TORCH:
    for mode in SOFT_MODES:
        CNN_STATES[mode] = {}
        for sd in CFG["cnn_seeds"]:
            t = time.time(); print(f"{WORD_NAME[mode]} seed {sd}")
            m = train_cnn(build_cnn(), X["train"], T_TRAIN[mode], CFG["epochs"], sd)
            CNN_STATES[mode][sd] = {k: v.detach().cpu().clone() for k, v in m.state_dict().items()}
            name = f"{WORD_NAME[mode]}__s{sd}"
            SCORES[name] = {k: predict_cnn(m, X[k]) for k in ("val", "dev")}; THR[name] = 0.5
            print(f"  done in {time.time() - t:.0f}s | dev PR-AUC {average_precision_score(Y['dev'], SCORES[name]['dev']):.3f}")
            del m
            if DEVICE.type == "cuda": torch.cuda.empty_cache()

CNN (words, raw similarity) seed 42
    epoch 1/10 loss 0.3999
    epoch 2/10 loss 0.3783
    epoch 3/10 loss 0.3692
    epoch 4/10 loss 0.3631
    epoch 5/10 loss 0.3566
    epoch 6/10 loss 0.3524
    epoch 7/10 loss 0.3472
    epoch 8/10 loss 0.3422
    epoch 9/10 loss 0.3373
    epoch 10/10 loss 0.3336
  done in 86s | dev PR-AUC 0.412
CNN (words, raw similarity) seed 43
    epoch 1/10 loss 0.4002
    epoch 2/10 loss 0.3792
    epoch 3/10 loss 0.3713
    epoch 4/10 loss 0.3633
    epoch 5/10 loss 0.3573
    epoch 6/10 loss 0.3521
    epoch 7/10 loss 0.3468
    epoch 8/10 loss 0.3421
    epoch 9/10 loss 0.3366
    epoch 10/10 loss 0.3327
  done in 85s | dev PR-AUC 0.395
CNN (words, raw similarity) seed 44
    epoch 1/10 loss 0.4019
    epoch 2/10 loss 0.3793
    epoch 3/10 loss 0.3702
    epoch 4/10 loss 0.3619
    epoch 5/10 loss 0.3569
    epoch 6/10 loss 0.3520
    epoch 7/10 loss 0.3462
    epoch 8/10 loss 0.3407
    epoch 9/10 loss 0.3353
    epoch 10/10 loss 0.3325
  done in 85s

## 6. CNN fine-tuned on signatures (words first, then HF dev)
Fine-tuning uses HF dev, so HF dev can no longer score it directly. Its selection score comes from signer-grouped 5-fold CV on HF dev (the existing `cv_seed42` folds): fine-tune on 4 folds, score the 5th.

In [11]:
if HAVE_TORCH:
    folds = splits.loc[dev.index, "cv_seed42"].to_numpy()
    for sd in CFG["cnn_seeds"]:
        oof = np.zeros(len(dev))
        for f in range(5):
            m = build_cnn(); m.load_state_dict(CNN_WORD[sd])
            tr_m = folds != f
            m = train_cnn(m, X["dev"][tr_m], Y["dev"][tr_m], CFG["finetune_epochs"], sd, max_lr=1e-3)
            oof[~tr_m] = predict_cnn(m, X["dev"][~tr_m])
            del m
        name = f"CNN (words + signature fine-tune)__s{sd}"
        SCORES[name] = {"dev": oof}; THR[name] = 0.5
        print(f"  {name} OOF done")

    epoch 1/8 loss 0.9400
    epoch 2/8 loss 0.9003
    epoch 3/8 loss 0.8547
    epoch 4/8 loss 0.7887
    epoch 5/8 loss 0.7613
    epoch 6/8 loss 0.7246
    epoch 7/8 loss 0.6971
    epoch 8/8 loss 0.7027
    epoch 1/8 loss 0.9233
    epoch 2/8 loss 0.8921
    epoch 3/8 loss 0.8428
    epoch 4/8 loss 0.7908
    epoch 5/8 loss 0.7536
    epoch 6/8 loss 0.7377
    epoch 7/8 loss 0.7168
    epoch 8/8 loss 0.7106
    epoch 1/8 loss 0.9079
    epoch 2/8 loss 0.8916
    epoch 3/8 loss 0.8449
    epoch 4/8 loss 0.8006
    epoch 5/8 loss 0.7680
    epoch 6/8 loss 0.7389
    epoch 7/8 loss 0.7155
    epoch 8/8 loss 0.6989
    epoch 1/8 loss 0.9747
    epoch 2/8 loss 0.9375
    epoch 3/8 loss 0.8921
    epoch 4/8 loss 0.8330
    epoch 5/8 loss 0.7834
    epoch 6/8 loss 0.7644
    epoch 7/8 loss 0.7416
    epoch 8/8 loss 0.7378
    epoch 1/8 loss 0.9410
    epoch 2/8 loss 0.9092
    epoch 3/8 loss 0.8597
    epoch 4/8 loss 0.8025
    epoch 5/8 loss 0.7755
    epoch 6/8 loss 0.7288
    epoch 7/

### Soft-target fine-tuning on signatures
Starts from each soft word model and fine-tunes on the **signature similarity** with the same mode, using the same `cv_seed42` signer folds (out-of-fold dev scores).

In [12]:
if HAVE_TORCH:
    folds = splits.loc[dev.index, "cv_seed42"].to_numpy()
    for mode in SOFT_MODES:
        for sd in CFG["cnn_seeds"]:
            oof = np.zeros(len(dev))
            for f in range(5):
                m = build_cnn(); m.load_state_dict(CNN_STATES[mode][sd])
                tr_m = folds != f
                m = train_cnn(m, X["dev"][tr_m], T_DEV[mode][tr_m], CFG["finetune_epochs"], sd, max_lr=1e-3)
                oof[~tr_m] = predict_cnn(m, X["dev"][~tr_m])
                del m
            name = f"{FT_NAME[mode]}__s{sd}"
            SCORES[name] = {"dev": oof}; THR[name] = 0.5
            print(f"  {name}: OOF dev PR-AUC {average_precision_score(Y['dev'], oof):.3f}")

    epoch 1/8 loss 0.5731
    epoch 2/8 loss 0.5623
    epoch 3/8 loss 0.5453
    epoch 4/8 loss 0.5346
    epoch 5/8 loss 0.5297
    epoch 6/8 loss 0.5235
    epoch 7/8 loss 0.5178
    epoch 8/8 loss 0.5171
    epoch 1/8 loss 0.5785
    epoch 2/8 loss 0.5643
    epoch 3/8 loss 0.5446
    epoch 4/8 loss 0.5365
    epoch 5/8 loss 0.5324
    epoch 6/8 loss 0.5266
    epoch 7/8 loss 0.5232
    epoch 8/8 loss 0.5199
    epoch 1/8 loss 0.5559
    epoch 2/8 loss 0.5390
    epoch 3/8 loss 0.5263
    epoch 4/8 loss 0.5170
    epoch 5/8 loss 0.5114
    epoch 6/8 loss 0.5059
    epoch 7/8 loss 0.5016
    epoch 8/8 loss 0.4996
    epoch 1/8 loss 0.5829
    epoch 2/8 loss 0.5646
    epoch 3/8 loss 0.5499
    epoch 4/8 loss 0.5376
    epoch 5/8 loss 0.5328
    epoch 6/8 loss 0.5231
    epoch 7/8 loss 0.5203
    epoch 8/8 loss 0.5179
    epoch 1/8 loss 0.5659
    epoch 2/8 loss 0.5489
    epoch 3/8 loss 0.5410
    epoch 4/8 loss 0.5274
    epoch 5/8 loss 0.5188
    epoch 6/8 loss 0.5131
    epoch 7/

## 7. Results on words-val and HF dev → choose the model

In [13]:
rows = []
for name, s in SCORES.items():
    base = name.split("__s")[0]
    r = {"model": base}
    for k in ("val", "dev"):
        if k in s:
            m = metrics(Y[k], s[k], THR[name])
            r[f"{k}_pr_auc"], r[f"{k}_roc_auc"], r[f"{k}_f1"] = m["pr_auc"], m["roc_auc"], m["f1"]
    rows.append(r)
res = pd.DataFrame(rows).groupby("model", sort=False).mean().round(3)
res = res.sort_values("dev_pr_auc", ascending=False)
print(f"chance PR-AUC: words-val {Y['val'].mean():.3f} | HF dev {Y['dev'].mean():.3f}")
display(res)

# per family on HF dev (ROC-AUC), to check the source-family confound
fam_rows = []
for name, s in SCORES.items():
    for fam in ("A", "B"):
        k = (dev["family"] == fam).to_numpy()
        if len(np.unique(Y["dev"][k])) == 2:
            fam_rows.append({"model": name.split("__s")[0], "family": fam,
                             "roc_auc": roc_auc_score(Y["dev"][k], s["dev"][k])})
display(pd.DataFrame(fam_rows).groupby(["model", "family"]).mean().unstack().round(3))
res.to_csv("word_models_selection.csv")

chance PR-AUC: words-val 0.502 | HF dev 0.293


,val_pr_auc,val_roc_auc,val_f1,dev_pr_auc,dev_roc_auc,dev_f1
model,,,,,,
CNN (words),0.825,0.836,0.764,0.457,0.657,0.481
"CNN (words, soft T0.05)",0.827,0.839,0.764,0.452,0.660,0.478
CNN (words + signature fine-tune),NaN,NaN,NaN,0.447,0.665,0.483
"CNN (words, soft T0.10)",0.825,0.839,0.766,0.432,0.655,0.475
"CNN (words + signature fine-tune, soft T0.05)",NaN,NaN,NaN,0.429,0.658,0.484
"CNN (words, raw similarity)",0.822,0.837,0.780,0.410,0.639,0.470
"CNN (words + signature fine-tune, soft T0.10)",NaN,NaN,NaN,0.403,0.644,0.475
"CNN (words + signature fine-tune, raw similarity)",NaN,NaN,NaN,0.401,0.633,0.484
SVM (RBF),0.769,0.795,0.729,0.385,0.595,0.420


roc_auc       
family                                                  A      B
model                                                           
CNN (words + signature fine-tune)                   0.596  0.639
CNN (words + signature fine-tune, raw similarity)   0.546  0.625
CNN (words + signature fine-tune, soft T0.05)       0.566  0.642
CNN (words + signature fine-tune, soft T0.10)       0.535  0.637
CNN (words)                                         0.578  0.651
CNN (words, raw similarity)                         0.544  0.635
CNN (words, soft T0.05)                             0.567  0.664
CNN (words, soft T0.10)                             0.553  0.666
Gradient boosting                                   0.563  0.581
Logistic (contrast only)                            0.505  0.500
Logistic (length only)                              0.593  0.515
Logistic regression                                 0.553  0.573
Majority baseline                                   0.500  0.500
Random forest                                       0.547  0.576
SVM (RBF)                                           0.592  0.592

### Is soft actually better than hard on HF dev?
Seed-ensemble scores (average of the 3 seeds), then a signer-grouped paired bootstrap of each soft variant minus its hard counterpart. This is informational: the locked rule below stays the pre-registered one (max HF-dev PR-AUC). But if the winner's Δ interval includes 0, say in the report that soft and hard are statistically tied on dev.

In [14]:
def ens(base):
    keys = [k for k in SCORES if k.split("__s")[0] == base and "dev" in SCORES[k]]
    return np.mean([SCORES[k]["dev"] for k in keys], axis=0) if keys else None

pairs = [(WORD_NAME[m], WORD_NAME["hard"]) for m in SOFT_MODES] + [(FT_NAME[m], FT_NAME["hard"]) for m in SOFT_MODES]
yd, gd = Y["dev"], dev["group"].to_numpy()
ug = np.unique(gd); by = [np.where(gd == g)[0] for g in ug]
boot = [np.concatenate([by[i] for i in r]) for r in np.random.default_rng(0).integers(0, len(ug), (2000, len(ug)))]
boot = [b for b in boot if len(np.unique(yd[b])) == 2]

rows = []
for soft, hard in pairs:
    s, h = ens(soft), ens(hard)
    if s is None or h is None: continue
    d = np.array([average_precision_score(yd[b], s[b]) - average_precision_score(yd[b], h[b]) for b in boot])
    rows.append({"soft": soft, "vs": hard,
                 "dev PR-AUC soft (ens)": average_precision_score(yd, s),
                 "dev PR-AUC hard (ens)": average_precision_score(yd, h),
                 "Δ [95% CI]": f"{np.mean(d):+.3f} [{np.percentile(d, 2.5):+.3f}, {np.percentile(d, 97.5):+.3f}]",
                 "P(soft better)": (d > 0).mean(),
                 "Spearman(score, similarity)": spearmanr(s, dev[SIG_SIM_COL]).statistic})
soft_vs_hard = pd.DataFrame(rows).round(3)
soft_vs_hard.to_csv("soft_vs_hard_dev.csv", index=False)
display(soft_vs_hard)

,soft,vs,dev PR-AUC soft (ens),dev PR-AUC hard (ens),Δ [95% CI],P(soft better),"Spearman(score, similarity)"
0,"CNN (words, raw similarity)",CNN (words),0.416,0.461,"-0.042 [-0.077, -0.006]",0.012,0.296
1,"CNN (words, soft T0.05)",CNN (words),0.459,0.461,"-0.001 [-0.027, +0.025]",0.465,0.334
2,"CNN (words, soft T0.10)",CNN (words),0.437,0.461,"-0.022 [-0.054, +0.012]",0.105,0.327
3,"CNN (words + signature fine-tune, raw similarity)",CNN (words + signature fine-tune),0.406,0.458,"-0.056 [-0.120, -0.003]",0.016,0.287
4,"CNN (words + signature fine-tune, soft T0.05)",CNN (words + signature fine-tune),0.437,0.458,"-0.019 [-0.048, +0.009]",0.088,0.315
5,"CNN (words + signature fine-tune, soft T0.10)",CNN (words + signature fine-tune),0.408,0.458,"-0.050 [-0.102, -0.004]",0.014,0.298


In [ ]:
# ---- LOCK THE CHOICE before the holdout ----
# Rule: highest HF-dev PR-AUC among learned models (baselines excluded).
BASELINES = {"Majority baseline", "Logistic (contrast only)", "Logistic (length only)"}
SELECTED = res.drop(index=[b for b in BASELINES if b in res.index]).index[0]
Path("selected_model.json").write_text(json.dumps(
    {"selected": SELECTED, "rule": "max HF-dev PR-AUC, baselines excluded",
     "locked_at": time.strftime("%Y-%m-%d %H:%M"), "config": CFG}, indent=1))
print("SELECTED:", SELECTED, "-> saved to selected_model.json")

## 8. Final holdout: HF test (run once)
Refits nothing that depends on test. The fine-tuned CNN is fine-tuned on **all** of HF dev here, then scored on test.

In [16]:
sel = json.loads(Path("selected_model.json").read_text())["selected"]   # fails if the choice was not locked

for name, (model, sets) in FITTED.items():
    test_set = {"Logistic (contrast only)": C, "Logistic (length only)": L}.get(name, F)
    SCORES[name]["test"], _ = mc.score_and_threshold(model, test_set["test"])
if HAVE_TORCH:
    for mode, states in CNN_STATES.items():            # "hard" plus every soft mode
        for sd in CFG["cnn_seeds"]:
            m = build_cnn(); m.load_state_dict(states[sd])
            SCORES[f"{WORD_NAME[mode]}__s{sd}"]["test"] = predict_cnn(m, X["test"])
            m = train_cnn(m, X["dev"], T_DEV[mode], CFG["finetune_epochs"], sd, max_lr=1e-3)
            SCORES[f"{FT_NAME[mode]}__s{sd}"]["test"] = predict_cnn(m, X["test"])
            torch.save(m.state_dict(), f"cnn_{mode}_finetuned_s{sd}.pt")   # for Tobacco800
            del m

def cluster_ci(y, s, groups, n=2000):
    r = np.random.default_rng(0); ug = np.unique(groups)
    by = [np.where(groups == g)[0] for g in ug]; pr = []
    for _ in range(n):
        idx = np.concatenate([by[i] for i in r.integers(0, len(ug), len(ug))])
        if len(np.unique(y[idx])) == 2: pr.append(average_precision_score(y[idx], s[idx]))
    return np.percentile(pr, [2.5, 97.5])

groups = test["group"].to_numpy(); rows = []
for name, s in SCORES.items():
    if "test" not in s: continue
    m = metrics(Y["test"], s["test"], THR[name]); lo, hi = cluster_ci(Y["test"], s["test"], groups)
    rows.append({"model": name.split("__s")[0], "pr_auc": m["pr_auc"], "pr_lo": lo, "pr_hi": hi,
                 "roc_auc": m["roc_auc"], "f1": m["f1"]})
final = pd.DataFrame(rows).groupby("model", sort=False).mean().round(3).sort_values("pr_auc", ascending=False)
final["selected"] = final.index == sel
final = final.join(res[["dev_pr_auc"]])
print(f"HF TEST (chance PR-AUC {Y['test'].mean():.3f}); CI = signer-group bootstrap; selected = {sel}")
display(final)
final.to_csv("word_models_holdout.csv")

    epoch 1/8 loss 0.9446
    epoch 2/8 loss 0.9198
    epoch 3/8 loss 0.8700
    epoch 4/8 loss 0.8086
    epoch 5/8 loss 0.7795
    epoch 6/8 loss 0.7623
    epoch 7/8 loss 0.7271
    epoch 8/8 loss 0.7199
    epoch 1/8 loss 0.9625
    epoch 2/8 loss 0.9291
    epoch 3/8 loss 0.8535
    epoch 4/8 loss 0.8241
    epoch 5/8 loss 0.7730
    epoch 6/8 loss 0.7406
    epoch 7/8 loss 0.7264
    epoch 8/8 loss 0.7152
    epoch 1/8 loss 0.9547
    epoch 2/8 loss 0.9210
    epoch 3/8 loss 0.8767
    epoch 4/8 loss 0.8537
    epoch 5/8 loss 0.8004
    epoch 6/8 loss 0.7797
    epoch 7/8 loss 0.7610
    epoch 8/8 loss 0.7454
    epoch 1/8 loss 0.5696
    epoch 2/8 loss 0.5557
    epoch 3/8 loss 0.5490
    epoch 4/8 loss 0.5313
    epoch 5/8 loss 0.5237
    epoch 6/8 loss 0.5222
    epoch 7/8 loss 0.5203
    epoch 8/8 loss 0.5168
    epoch 1/8 loss 0.5771
    epoch 2/8 loss 0.5609
    epoch 3/8 loss 0.5427
    epoch 4/8 loss 0.5355
    epoch 5/8 loss 0.5234
    epoch 6/8 loss 0.5173
    epoch 7/

,pr_auc,pr_lo,pr_hi,roc_auc,f1,selected,dev_pr_auc
model,,,,,,,
"CNN (words + signature fine-tune, soft T0.05)",0.621,0.392,0.826,0.816,0.630,False,0.429
"CNN (words + signature fine-tune, soft T0.10)",0.619,0.400,0.807,0.812,0.630,False,0.403
"CNN (words, soft T0.05)",0.598,0.385,0.782,0.792,0.591,False,0.452
"CNN (words, soft T0.10)",0.590,0.375,0.771,0.782,0.600,False,0.432
CNN (words + signature fine-tune),0.583,0.365,0.793,0.790,0.605,False,0.447
CNN (words),0.575,0.367,0.757,0.779,0.589,True,0.457
"CNN (words + signature fine-tune, raw similarity)",0.557,0.362,0.757,0.781,0.612,False,0.401
"CNN (words, raw similarity)",0.545,0.359,0.727,0.766,0.598,False,0.410
Random forest,0.436,0.306,0.609,0.645,0.497,False,0.375
